In [ ]:
# ===========================================================================
# The exchange rate: how many LLM-labelled sentences buy one human-labelled one.
#
# Three point estimates at 2248 sentences say human 0.785, LLM labels 0.657,
# synthetic 0.462. That answers "which is better" but not "how much cheaper does
# labelling have to be to be worth it", which is the question a reader has.
#
# The top two arms are the same sentences with the same gold entity spans, so the
# only thing varying between them is where the labels came from. The synthetic arm
# is a different text distribution entirely and is here as the floor.
#
# Training only. No API calls. Everything is already on disk.
#
# Cost: 3 arms x 5 budgets x seeds. Small budgets train in well under a minute, so
# the whole sweep is roughly an hour. Resumable, so it can be interrupted.
# ===========================================================================
import glob
import json
import os

import pandas as pd

from ddi import label_real
from ddi.data import build_human
from ddi.experiment import log_run
from ddi.manifest import load_dataset
from ddi.mixing import subsample_by_sentence
from ddi.train import train_and_eval

DATASET_ID_V18_LOW = "20260825-095703-67c1fa"
LABEL_VERDICTS = "datasets/instances/raw/label-high-c8.jsonl"

# Below 500 sentences the variance is large enough that 3 seeds is not enough to
# rank anything, and the left end is exactly where the curves separate. Above it,
# 3 is fine because the arms are far apart.
BUDGETS = [100, 250, 500, 1000, 2248]
SEEDS_SMALL = [0, 1, 2, 3, 4]
SEEDS_LARGE = [0, 1, 2]
SMALL = 500

BASE = {"model_name": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
        "epochs": 3, "lr": 2e-5, "batch_size": 32, "max_length": 256,
        "neg_ratio": None, "render_mode": "markers"}

train, dev, val = build_human()

# The human pool is filtered the same way the labelling arm was, so the two arms
# draw from the same sentences. Without this the human arm keeps the eight
# enumeration monsters that carry 43% of instances and almost no positives, and the
# comparison stops being like for like.
per_sent = {}
for r in train:
    per_sent[r["sent_id"]] = per_sent.get(r["sent_id"], 0) + 1
train_f = [r for r in train if per_sent[r["sent_id"]] < 190]

label_inst, dropped = label_real.to_instances(train, LABEL_VERDICTS)
v18_low, _ = load_dataset(DATASET_ID_V18_LOW)

pools = {
    "human": train_f,
    "llm-labels": label_inst,
    "synthetic": v18_low,
}
for name, pool in pools.items():
    n_sent = len({r["sent_id"] for r in pool})
    pos = sum(1 for r in pool if r["label"] != "NONE") / len(pool)
    print(f"{name:12} {n_sent:>6} sentences  {len(pool):>7} instances  "
          f"pos rate {pos:.3f}")
print(f"({dropped} labelled sentences dropped for shape mismatch)\n")


def arm_name(pool, budget):
    return f"xr-{pool}-{budget}"


def load_done(runs_dir="runs"):
    """Skip anything already logged. Keyed on the arm name encoding pool and budget,
    so the sweep can be interrupted and resumed without repeating GPU time."""
    done = {}
    for path in sorted(glob.glob(os.path.join(runs_dir, "*.json"))):
        try:
            r = json.load(open(path))
        except Exception:
            continue
        if isinstance(r, list):
            continue
        cfg, m = r.get("config", {}), r.get("metrics", {})
        name, seed = cfg.get("dataset"), cfg.get("seed")
        if isinstance(name, str) and name.startswith("xr-") and seed is not None \
                and "micro_f1_pos" in m:
            done[(name, seed)] = {"pool": name.split("-")[1],
                                  "budget": int(name.split("-")[-1]),
                                  "seed": seed, "f1": m["micro_f1_pos"],
                                  "p": m["micro_p_pos"], "r": m["micro_r_pos"]}
    return done


done = load_done()
print(f"{len(done)} runs recovered from the log\n")
rows = list(done.values())

for budget in BUDGETS:
    seeds = SEEDS_SMALL if budget < SMALL else SEEDS_LARGE
    for pool_name, pool in pools.items():
        for seed in seeds:
            name = arm_name(pool_name, budget)
            if (name, seed) in done:
                continue
            recs = subsample_by_sentence(pool, budget, seed=seed)
            if not recs:
                continue
            cfg = {**BASE, "seed": seed, "dataset": name}
            m = train_and_eval(cfg, recs, dev)
            log_run(cfg, m, notes=f"exchange rate, {pool_name} at {budget} sentences")
            rows.append({"pool": pool_name, "budget": budget, "seed": seed,
                         "f1": m["micro_f1_pos"], "p": m["micro_p_pos"],
                         "r": m["micro_r_pos"]})
            print(f"{pool_name:12} {budget:>5} sentences seed={seed} "
                  f"f1={m['micro_f1_pos']:.3f}")

    df = pd.DataFrame(rows)
    piv = (df.groupby(["budget", "pool"])["f1"].agg(["mean", "std", "count"])
             .round(3).unstack("pool"))
    print(f"\n===== through budget {budget} =====")
    print(piv.to_string())
    print()

# ---------------------------------------------------------------------------
# The exchange rate itself: for each human budget, how many LLM-labelled sentences
# reach the same F1. Read off by interpolating the LLM curve, so it is only defined
# where the LLM curve actually spans the human score.
# ---------------------------------------------------------------------------
df = pd.DataFrame(rows)
means = df.groupby(["pool", "budget"])["f1"].mean()


def rate_table():
    llm = means.get("llm-labels")
    hum = means.get("human")
    if llm is None or hum is None:
        return
    xs, ys = list(llm.index), list(llm.values)
    print(f"{'human sentences':>16} {'human F1':>9} {'LLM sentences to match':>24}")
    for b in hum.index:
        target = hum[b]
        got = None
        for i in range(len(xs) - 1):
            lo, hi = ys[i], ys[i + 1]
            if (lo - target) * (hi - target) <= 0 and hi != lo:
                frac = (target - lo) / (hi - lo)
                got = xs[i] + frac * (xs[i + 1] - xs[i])
                break
        if got is None:
            note = "beyond the LLM curve" if target > max(ys) else "below its start"
            print(f"{b:>16} {target:>9.3f} {note:>24}")
        else:
            print(f"{b:>16} {target:>9.3f} {got:>21.0f}  "
                  f"({got / b:.1f}x)")


rate_table()
print("""
Reading it.

The ratio is the exchange rate: how many LLM-labelled sentences it takes to match one
human-labelled one at that budget. If it is roughly flat across budgets, one number
describes the trade and the cost argument is simple. If it grows, LLM labels have a
ceiling human labels do not, and the honest claim is that labelling substitutes only
up to some budget.

Where the human curve goes above the top of the LLM curve, no amount of LLM labelling
matches it, and that budget is the answer on its own. Note the LLM arm cannot be
extended past 2402 sentences without more corpus, which is the real bound on that arm
and worth stating rather than treating as a sampling limit.

The synthetic arm is the floor. If it tracks the LLM arm at small budgets and only
falls behind later, the generator is competitive exactly where annotation is scarce,
which is the regime the mixing curve already identified as the one that matters.
""")